In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [58]:
pd.set_option('display.max_columns', None)
df=pd.read_csv('Customer-Churn-Records.csv')
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339


In [59]:
df.columns=df.columns.str.lower().str.replace(' ', '_')
print(df)

      rownumber  customerid    surname  creditscore geography  gender  age  \
0             1    15634602   Hargrave          619    France  Female   42   
1             2    15647311       Hill          608     Spain  Female   41   
2             3    15619304       Onio          502    France  Female   42   
3             4    15701354       Boni          699    France  Female   39   
4             5    15737888   Mitchell          850     Spain  Female   43   
...         ...         ...        ...          ...       ...     ...  ...   
9995       9996    15606229   Obijiaku          771    France    Male   39   
9996       9997    15569892  Johnstone          516    France    Male   35   
9997       9998    15584532        Liu          709    France  Female   36   
9998       9999    15682355  Sabbatini          772   Germany    Male   42   
9999      10000    15628319     Walker          792    France  Female   28   

      tenure    balance  numofproducts  hascrcard  isactivememb

In [60]:
columns_to_drop = ['rownumber', 'customerid', 'surname']
df = df.drop(columns=columns_to_drop)
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values[missing_values > 0])
display(df.head())

Missing Values:
 Series([], dtype: int64)


,creditscore,geography,gender,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,complain,satisfaction_score,card_type,point_earned
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [61]:
print(f"Dataset Shape: {df.shape}")
 

Dataset Shape: (10000, 15)


(Data Preparation & EDA).

In [62]:
from sklearn.model_selection import train_test_split
df_full_train, df_test=train_test_split(df_full_train, test_size=0.25, random_state=42)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train=df_train['exited'].values
y_val=df_val['exited'].values
y_test=df_test['exited'].values
del df_train['exited']
del df_val['exited']
del df_test['exited']
print("Train shape:", df_train.shape)
print("Val shape:", df_val.shape)    
print("Test shape:", df_test.shape)  

Train shape: (2531, 14)
Val shape: (844, 14)
Test shape: (1125, 14)


Feature Analysis


In [63]:
from sklearn.metrics import mutual_info_score

numerical = ['creditscore', 'age', 'tenure', 'balance', 'numofproducts', 'estimatedsalary', 'point_earned']        
categorical = ['geography', 'gender', 'hascrcard', 'isactivemember', 'complain', 'satisfaction_score', 'card_type']
# DictVectorizer
for col in categorical:
    df_full_train[col]=df_full_train[col].astype(str)
    df_train[col]=df_train[col].astype(str)
    df_val[col]=df_val[col].astype(str)
print('==== Numerical Feature Correlation ====') 

# (exited)                                                                                                 
correlation=df_full_train[numerical].apply(lambda x: x.corr(df_full_train['exited'])).sort_values(key=abs, ascending=False)
print(correlation)
#==============
def calculate_mi(series):                                                              
    return mutual_info_score(series, df_full_train['exited'])                          
                                                                                       
                                                                                       
mi_scores = df_full_train[categorical].apply(calculate_mi).sort_values(ascending=False)
print(mi_scores)                                                                       

==== Numerical Feature Correlation ====
age                0.287261
balance            0.118152
numofproducts     -0.081409
creditscore       -0.013271
estimatedsalary    0.006945
tenure            -0.003572
point_earned      -0.000876
dtype: float64
complain              0.498516
geography             0.016420
isactivemember        0.009956
gender                0.003248
card_type             0.000720
satisfaction_score    0.000227
hascrcard             0.000107
dtype: float64


Model Training

In [64]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. دمج الميزات الفئوية والرقمية التي سنستخدمها للتدريب
features = categorical + numerical

# 2. تحويل بيانات التدريب إلى قواميس (Dictionaries) ثم تشفيرها
dv = DictVectorizer(sparse=False)
train_dict = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

# تحويل بيانات التحقق بنفس الطريقة
val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

# 3. بناء وتدريب نموذج Logistic Regression
# نستخدم solver='liblinear' لأنه مناسب لمجموعات البيانات الصغيرة والمتوسطة
model_lr = LogisticRegression(solver='liblinear', random_state=42)
model_lr.fit(X_train, y_train)

# 4. التنبؤ بالاحتمالات (Probabilities) على بيانات التحقق
y_pred_proba = model_lr.predict_proba(X_val)[:, 1]

# 5. التنبؤ بالفئات المباشرة (باستخدام حد القطع الافتراضي 0.5)
y_pred_class = model_lr.predict(X_val)

# 6. التقييم الأولي (Accuracy و ROC AUC) كما في الوحدة 4
accuracy = accuracy_score(y_val, y_pred_class)
roc_auc = roc_auc_score(y_val, y_pred_proba)

print(f"Logistic Regression - Validation Accuracy: {accuracy:.4f}")
print(f"Logistic Regression - Validation ROC AUC: {roc_auc:.4f}")

Logistic Regression - Validation Accuracy: 0.7749
Logistic Regression - Validation ROC AUC: 0.6772


XGBoost

In [65]:
import xgboost as xgb
import re

# 1. تنظيف أسماء الميزات (Feature Names)
# XGBoost لا يقبل رموز مثل '[', ']', '<' في أسماء الأعمدة الناتجة عن DictVectorizer
feature_names = dv.get_feature_names_out().tolist()
clean_feature_names = [re.sub(r'[\[\]<>=]', '_', f) for f in feature_names]

# 2. إنشاء هيكل البيانات DMatrix المخصص لـ XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=clean_feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=clean_feature_names)

# 3. إعداد معلمات النموذج (Hyperparameters)
# بناءً على مفاهيم الوحدة السادسة، نختار معلمات مبدئية جيدة
xgb_params = {
    'eta': 0.1,                     # معدل التعلم (Learning Rate)
    'max_depth': 4,                 # أقصى عمق للشجرة
    'min_child_weight': 1,          # الحد الأدنى لوزن الورقة
    'objective': 'binary:logistic', # لأنها مشكلة تصنيف ثنائي (Churn)
    'eval_metric': 'auc',           # نستخدم مساحة ROC للتقييم
    'nthread': 8,
    'seed': 42,
    'verbosity': 1,
}

# 4. تدريب النموذج مع تتبع الأداء (Watchlist)
watchlist = [(dtrain, 'train'), (dval, 'val')]

model_xgb = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,           # عدد الأشجار
    evals=watchlist,
    verbose_eval=10                # طباعة النتيجة كل 10 خطوات
)

# 5. التنبؤ والتقييم النهائي
y_pred_xgb = model_xgb.predict(dval)
roc_auc_xgb = roc_auc_score(y_val, y_pred_xgb)

print("\n" + "="*40)
print(f"XGBoost - Validation ROC AUC: {roc_auc_xgb:.4f}")
print("="*40)

[0]	train-auc:0.99901	val-auc:0.99924
[10]	train-auc:0.99925	val-auc:0.99921
[20]	train-auc:0.99973	val-auc:0.99919
[30]	train-auc:0.99996	val-auc:0.99988
[40]	train-auc:0.99998	val-auc:0.99994
[50]	train-auc:1.00000	val-auc:0.99994
[60]	train-auc:1.00000	val-auc:0.99994
[70]	train-auc:1.00000	val-auc:0.99994
[80]	train-auc:1.00000	val-auc:0.99997
[90]	train-auc:1.00000	val-auc:0.99996
[99]	train-auc:1.00000	val-auc:0.99995

XGBoost - Validation ROC AUC: 1.0000
